# 02 Data Validation
Bronze tables -> Read into PySpark -> Validation -> Validation Report

## Read Bronze Tables

In [0]:
stores_df = spark.table("stores_bronze")
products_df = spark.table("products_bronze")
suppliers_df = spark.table("suppliers_bronze")
sales_df = spark.table("sales_bronze")
inventory_df = spark.table("inventory_bronze")
purchase_orders_df = spark.table("purchase_orders_bronze")

datasets = {
    "stores": stores_df,
    "products": products_df,
    "suppliers": suppliers_df,
    "sales": sales_df,
    "inventory": inventory_df,
    "purchase_orders": purchase_orders_df
}


### Row count

In [0]:
for name, df in datasets.items():
    print(f"{name:20} : {df.count()} rows")

### Schema Check

In [0]:
for name, df in datasets.items():
    print("=" * 60)
    print(name.upper())
    df.printSchema()

### Missing value check

In [0]:
def check_nulls(df, table_name):
    null_counts = df.select([
        F.count(F.when(F.col(c).isNull(), c)).alias(c)
        for c in df.columns
    ])

    print(f"\n{table_name}")
    null_counts.show()

print("Null counts")
for name, df in datasets.items():
    check_nulls(df, name)

## Missing Manager Name

In [0]:
missing_manager_name = (
    stores_df
    .filter(F.col("manager_name").isNull())
)

missing_manager_name.show()
print("Missing manager names:", missing_manager_name.count())

## Missing Supplier Contact Person

In [0]:
missing_contact_person = (
    suppliers_df
    .filter(F.col("contact_person").isNull())
)

missing_contact_person.show()
print("Missing contact persons:", missing_contact_person.count())

### Duplicate validation

In [0]:
print("Exact duplicate rows:")
print(sales_df.count() - sales_df.dropDuplicates().count())

Same `transaction_id` values but the same sale details

In [0]:
uplicate_sales_records = (
    sales_df
    .groupBy(
        "store_id",
        "product_id",
        "quantity_sold",
        "unit_price",
        "total_amount",
        "sale_date"
    )
    .count()
    .filter(F.col("count") > 1)
)

duplicate_sales_records.show()
print("Duplicate sales record groups:", duplicate_sales_records.count())

In [0]:
%sql 
select *
from sales_bronze
where store_id=3 and product_id=1010 and quantity_sold=9

## Duplicate Primary Keys

Each table should have a unique primary key.

In [0]:
primary_keys = {
    "stores": ("stores_df", "store_id"),
    "products": ("products_df", "product_id"),
    "suppliers": ("suppliers_df", "supplier_id"),
    "sales": ("sales_df", "transaction_id"),
    "inventory": ("inventory_df", "inventory_id"),
    "purchase_orders": ("purchase_orders_df", "po_id")
}

pk_duplicate_counts = {}

for table_name, (df_name, pk) in primary_keys.items():
    df = globals()[df_name]

    count = (
        df.groupBy(pk)
          .count()
          .filter(F.col("count") > 1)
          .count()
    )

    pk_duplicate_counts[table_name] = count
    print(f"{table_name:20} : {count} duplicate {pk} groups")

## Inventory Business Key

There should be only one inventory record for each store-product combination.

In [0]:
inventory_duplicate_check = (
    inventory_df
    .groupBy("store_id", "product_id")
    .count()
    .filter(F.col("count") > 1)
)

inventory_duplicate_check.show()
print(
    "Duplicate inventory business-key groups:",
    inventory_duplicate_check.count()
)

### Product price validation

In [0]:
invalid_product_prices = (
    products_df
    .filter(
        (F.col("unit_price").isNull()) |
        (F.col("unit_price") <= 0)
    )
)

invalid_product_prices.show()
print("Invalid product prices:", invalid_product_prices.count())

### Sales price validation

Product price should be greater than zero

In [0]:
invalid_sales_prices = (
    sales_df
    .filter(
        (F.col("unit_price").isNull()) |
        (F.col("unit_price") <= 0)
    )
)

invalid_sales_prices.show()

## Inventory Quantity Validation

Zero stock is valid because it represents an out-of-stock product. Negative stock is invalid.

In [0]:
invalid_inventory_quantity = (
    inventory_df
    .filter(
        F.col("quantity_on_hand").isNull() |
        (F.col("quantity_on_hand") < 0)
    )
)

invalid_inventory_quantity.show()
print("Invalid inventory quantities:", invalid_inventory_quantity.count())

### Sales quantity validation

In [0]:
invalid_sales_quantity = (
    sales_df
    .filter(
        F.col("quantity_sold").isNull() |
        (F.col("quantity_sold") <= 0)
    )
)

invalid_sales_quantity.show()

In [0]:
invalid_inventory_quantity = (
    inventory_df
    .filter(
        F.col("quantity_on_hand").isNull() |
        (F.col("quantity_on_hand") < 0)
    )
)

invalid_inventory_quantity.show()

### Total amount validation
`total_amount` should equal `quantity_sold * unit_price`.

In [0]:
invalid_total_amount = (
    sales_df
    .filter(
        F.abs(
            F.col("total_amount") -
            (F.col("quantity_sold") * F.col("unit_price"))
        ) > 0.01
    )
)

invalid_total_amount.show()

### Foreign key `validation`

In [0]:
invalid_sales_store = (
    sales_df
    .join(stores_df.select("store_id"), on="store_id", how="left_anti")
)

invalid_sales_product = (
    sales_df
    .join(products_df.select("product_id"), on="product_id", how="left_anti")
)

invalid_product_supplier = (
    products_df
    .join(suppliers_df.select("supplier_id"), on="supplier_id", how="left_anti")
)

invalid_inventory_store = (
    inventory_df
    .join(stores_df.select("store_id"), on="store_id", how="left_anti")
)

invalid_inventory_product = (
    inventory_df
    .join(products_df.select("product_id"), on="product_id", how="left_anti")
)

invalid_po_supplier = (
    purchase_orders_df
    .join(suppliers_df.select("supplier_id"), on="supplier_id", how="left_anti")
)

print("Invalid sales store IDs:", invalid_sales_store.count())
print("Invalid sales product IDs:", invalid_sales_product.count())
print("Invalid product supplier IDs:", invalid_product_supplier.count())
invalid_product_supplier.show()
print("Invalid inventory store IDs:", invalid_inventory_store.count())
print("Invalid inventory product IDs:", invalid_inventory_product.count())
print("Invalid PO supplier IDs:", invalid_po_supplier.count())

## Purchase Order Date Validation

A delivery date cannot be earlier than its order date. A NULL delivery date is allowed because it represents a pending order in the generated dataset.

In [0]:
invalid_po_dates = (
    purchase_orders_df
    .filter(
        F.col("delivery_date").isNotNull() &
        (F.col("delivery_date") < F.col("order_date"))
    )
)

invalid_po_dates.show()
print("Invalid PO date relationships:", invalid_po_dates.count())

### Inventory business key `validation`

In [0]:
inventory_duplicate_check = (
    inventory_df
    .groupBy("store_id", "product_id")
    .count()
    .filter(F.col("count") > 1)
)

inventory_duplicate_check.show()

print(
    "Duplicate inventory business-key groups:",
    inventory_duplicate_check.count()
)

In [0]:
invalid_po_dates = (
    purchase_orders_df
    .filter(
        F.col("delivery_date").isNotNull() &
        (F.col("delivery_date") < F.col("order_date"))
    )
)

invalid_po_dates.show()

## Date Range Validation

The project data covers January to December 2024.

In [0]:
DATA_START = "2024-01-01"
DATA_END = "2024-12-31"

invalid_sales_dates = (
    sales_df
    .filter(
        (F.col("sale_date") < DATA_START) |
        (F.col("sale_date") > DATA_END)
    )
)

invalid_po_order_dates = (
    purchase_orders_df
    .filter(
        (F.col("order_date") < DATA_START) |
        (F.col("order_date") > DATA_END)
    )
)

invalid_po_delivery_dates = (
    purchase_orders_df
    .filter(
        F.col("delivery_date").isNotNull() &
        (
            (F.col("delivery_date") < DATA_START) |
            (F.col("delivery_date") > DATA_END)
        )
    )
)

print("Invalid sales dates:", invalid_sales_dates.count())
invalid_sales_dates.show()
print("Invalid PO order dates:", invalid_po_order_dates.count())
print("Invalid PO delivery dates:", invalid_po_delivery_dates.count())
invalid_po_delivery_dates.show()

## Phone Format Validation

The cleaning stage will standardize phone numbers. Here we only identify values that are not in the chosen 10-digit format.

In [0]:
inconsistent_phone_format = (
    suppliers_df
    .filter(
        ~F.col("phone").rlike(r"^\d{10}$")
    )
)

inconsistent_phone_format.show(truncate=False)
print(
    "Inconsistent phone formats:",
    inconsistent_phone_format.count()
)

## Validation summary

In [0]:
validation_results = [
    ("Missing manager names", missing_manager_name.count()),
    ("Missing contact persons", missing_contact_person.count()),

    ("Duplicate sales records", duplicate_sales_records.count()),
    ("Duplicate inventory business keys", inventory_duplicate_check.count()),

    ("Duplicate stores PKs", pk_duplicate_counts["stores"]),
    ("Duplicate products PKs", pk_duplicate_counts["products"]),
    ("Duplicate suppliers PKs", pk_duplicate_counts["suppliers"]),
    ("Duplicate sales PKs", pk_duplicate_counts["sales"]),
    ("Duplicate inventory PKs", pk_duplicate_counts["inventory"]),
    ("Duplicate purchase order PKs", pk_duplicate_counts["purchase_orders"]),

    ("Invalid product prices", invalid_product_prices.count()),
    ("Invalid sales prices", invalid_sales_prices.count()),

    ("Invalid sales quantities", invalid_sales_quantity.count()),
    ("Invalid inventory quantities", invalid_inventory_quantity.count()),

    ("Invalid total amounts", invalid_total_amount.count()),

    ("Invalid sales store IDs", invalid_sales_store.count()),
    ("Invalid sales product IDs", invalid_sales_product.count()),
    ("Invalid product supplier IDs", invalid_product_supplier.count()),
    ("Invalid inventory store IDs", invalid_inventory_store.count()),
    ("Invalid inventory product IDs", invalid_inventory_product.count()),
    ("Invalid PO supplier IDs", invalid_po_supplier.count()),

    ("Invalid PO date relationships", invalid_po_dates.count()),
    ("Invalid sales dates", invalid_sales_dates.count()),
    ("Invalid PO order dates", invalid_po_order_dates.count()),
    ("Invalid PO delivery dates", invalid_po_delivery_dates.count()),

    ("Inconsistent phone formats", inconsistent_phone_format.count())
]

validation_summary = spark.createDataFrame(
    validation_results,
    ["check_name", "issue_count"]
)

validation_summary.show(truncate=False)

In [0]:
validation_results